# 01: Translation run

Iterates every gold-standard query × dialect pair, drives `SQLTranslator` through the generate-validate-fix loop, and serialises the result to `evaluation/outputs/records.json`. Every downstream notebook consumes that file.

This is the **only** notebook that calls the LLM. The other notebooks operate on the cached records. Keep this in mind when you tweak metric code, you don't need to re-run this notebook each time.

Configuration choices baked in here:

- **Temperature 0.0** (overrides the library default of 0.1) so the LLM is deterministic. Critical for reproducible numbers in the thesis.
- **Syntax validation only** (`make_validator(..., "syntax")`). Server-side validation would catch additional hallucination categories but would also block this notebook on Neo4j / ArangoDB connectivity; that's reserved for `05_execution_metrics`.
- **Max 3 iterations** of the fix loop (the library default).
- **Anthropic backend** via `config/models/anthropic.yaml`. To compare against a weaker baseline, change `MODEL_CONFIG_PATH` below to point at `config/models/ollama.yaml` and rerun; save to a separate records file (e.g. `records_ollama.json`).

Token counts are scraped from the Anthropic SDK by attaching a `logging.Handler` to `rows2graph.llm.anthropic`. The library already emits `"Anthropic call: input=N output=M tokens (model=...)"` at INFO level; we parse those messages with a regex so the library itself doesn't need a hook.

In [ ]:
from __future__ import annotations

import json
import logging
import re
import sys
from dataclasses import dataclass, field
from pathlib import Path
from time import perf_counter

import yaml

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

DATASETS_DIR = REPO_ROOT / "evaluation" / "datasets"
MAPPINGS_DIR = REPO_ROOT / "config" / "mappings"
OUTPUTS_DIR = REPO_ROOT / "evaluation" / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

MODEL_CONFIG_PATH = REPO_ROOT / "config" / "models" / "anthropic.yaml"
RECORDS_OUT = OUTPUTS_DIR / "records.json"
MAX_ITERATIONS = 3

# Set to a list of query IDs (e.g. ["ldbc_q01", "ldbc_q04"]) to smoke-test a slice
# before paying for a full run. None = translate everything.
SUBSET: list[str] | None = None

# If True, skip queries whose (query_id, dialect) is already in RECORDS_OUT.
# Lets you resume after a failure / interruption without re-spending tokens.
RESUME = True

print(f"Model config:  {MODEL_CONFIG_PATH}")
print(f"Records out:   {RECORDS_OUT}")
print(f"Max iterations: {MAX_ITERATIONS}")
print(f"Subset:        {SUBSET if SUBSET else 'ALL'}")
print(f"Resume:        {RESUME}")

## Library imports and a deterministic Anthropic config

We load the model config from YAML and then override `temperature=0.0`. The library exposes `AnthropicConfig` as a Pydantic model, so this is a `model_copy(update=...)` call rather than a YAML edit: no on-disk state changes for an eval run.

In [ ]:
from rows2graph import (
    AnthropicConfig,
    OllamaConfig,
    SchemaMapping,
    SQLTranslator,
    load_model_config,
    make_llm,
    make_target,
    make_validator,
)

raw_model_cfg = load_model_config(MODEL_CONFIG_PATH)
model_cfg = raw_model_cfg.model_copy(update={"temperature": 0.0})
print(f"Model: {model_cfg.provider} / {model_cfg.model}  temperature={model_cfg.temperature}")
MODEL_NAME = f"{model_cfg.provider}/{model_cfg.model}"

## Token-count log scraper

The Anthropic client logs `"Anthropic call: input=N output=M tokens (model=...)"` after every API call. We parse those messages with a list-handler so each translation can read off the cumulative tokens it consumed without having to monkey-patch the SDK.

In [ ]:
TOKEN_LINE = re.compile(r"Anthropic call: input=(\d+) output=(\d+) tokens")


class TokenScraper(logging.Handler):
    """Captures per-call (input_tokens, output_tokens) tuples from the Anthropic log."""

    def __init__(self) -> None:
        super().__init__(level=logging.INFO)
        self.calls: list[tuple[int, int]] = []

    def emit(self, record: logging.LogRecord) -> None:
        m = TOKEN_LINE.search(record.getMessage())
        if m:
            self.calls.append((int(m.group(1)), int(m.group(2))))

    def reset(self) -> tuple[int, int]:
        """Pop the accumulated counts and return (input_total, output_total)."""
        in_total = sum(i for i, _ in self.calls)
        out_total = sum(o for _, o in self.calls)
        self.calls.clear()
        return in_total, out_total


anthropic_logger = logging.getLogger("rows2graph.llm.anthropic")
anthropic_logger.setLevel(logging.INFO)
scraper = TokenScraper()
anthropic_logger.addHandler(scraper)
print("Token scraper attached to rows2graph.llm.anthropic")

## Load datasets and build the work list

We flatten gold-query × dialect into one work item per row. Each item is what one translator invocation will produce.

In [ ]:
@dataclass
class WorkItem:
    schema_name: str
    query_id: str
    difficulty: str
    sql_features: list[str]
    sql: str
    dialect: str  # "cypher" | "aql"
    expected_query: str
    mapping_path: Path


work: list[WorkItem] = []
for ds_path in sorted(DATASETS_DIR.glob("*.yaml")):
    data = yaml.safe_load(ds_path.read_text())
    schema_name = data["schema"]
    mapping_path = MAPPINGS_DIR / f"{schema_name}.yaml"
    for q in data["queries"]:
        if SUBSET is not None and q["id"] not in SUBSET:
            continue
        for dialect, key in (("cypher", "expected_cypher"), ("aql", "expected_aql")):
            work.append(
                WorkItem(
                    schema_name=schema_name,
                    query_id=q["id"],
                    difficulty=q["difficulty"],
                    sql_features=q.get("sql_features", []),
                    sql=q["sql"],
                    dialect=dialect,
                    expected_query=q[key],
                    mapping_path=mapping_path,
                )
            )
print(f"Total work items: {len(work)}  ({sum(1 for w in work if w.dialect == 'cypher')} cypher, {sum(1 for w in work if w.dialect == 'aql')} aql)")

## Resume support

If `RESUME=True` and `records.json` already exists, we skip work items whose `(query_id, dialect)` pair has been done. This lets you abort a long run (Ctrl-C) and continue without losing prior translations or paying for them again.

In [ ]:
existing_records: list[dict] = []
done_keys: set[tuple[str, str]] = set()
if RESUME and RECORDS_OUT.exists():
    existing_records = json.loads(RECORDS_OUT.read_text())
    done_keys = {(r["query_id"], r["dialect"]) for r in existing_records}
    print(f"Resume: {len(existing_records)} records already on disk, {len(done_keys)} (query_id, dialect) pairs covered.")
else:
    print("No resume: starting from scratch.")

todo = [w for w in work if (w.query_id, w.dialect) not in done_keys]
print(f"To translate this run: {len(todo)} item(s).")

## The translation loop

For each work item:

1. Load the per-schema mapping (cached so we don't re-parse the YAML 28 times).
2. Construct an `SQLTranslator` with a fresh LLM client + target + syntax validator.
3. Reset the token scraper, call `translate(sql)`, read off the tokens consumed.
4. Append one record to `records.json` after **every** translation; that way a crash mid-run preserves the work that has already happened.

We construct a fresh translator per item (instead of reusing one across all items) because the translator's chat history would otherwise pollute subsequent prompts. The library exposes `close()` and supports the context-manager protocol; using it inside `with` ensures Neo4j drivers (relevant once we layer server validation on top) get released.

In [ ]:
def write_records(records: list[dict]) -> None:
    RECORDS_OUT.write_text(json.dumps(records, indent=2, default=str))


mapping_cache: dict[Path, SchemaMapping] = {}


def mapping_for(path: Path) -> SchemaMapping:
    if path not in mapping_cache:
        mapping_cache[path] = SchemaMapping.from_yaml(path)
    return mapping_cache[path]


records: list[dict] = list(existing_records)
for idx, item in enumerate(todo, start=1):
    print(f"[{idx:3d}/{len(todo)}] {item.query_id} → {item.dialect}", end=" ", flush=True)
    mapping = mapping_for(item.mapping_path)
    llm = make_llm(model_cfg)
    target = make_target(item.dialect)
    validator = make_validator(item.dialect, "syntax")

    scraper.reset()
    t0 = perf_counter()
    error_message: str | None = None
    result = None
    try:
        with SQLTranslator(mapping, llm, target, validator, max_iterations=MAX_ITERATIONS) as translator:
            result = translator.translate(item.sql)
    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        print(f"ERROR ({error_message})")
    duration = perf_counter() - t0
    input_tokens, output_tokens = scraper.reset()

    rec = {
        "query_id": item.query_id,
        "schema_name": item.schema_name,
        "dialect": item.dialect,
        "model_name": MODEL_NAME,
        "difficulty": item.difficulty,
        "sql_features": item.sql_features,
        "sql": item.sql,
        "expected_query": item.expected_query,
        "generated_query": result.generated_query if result else None,
        "validation_passed": bool(result and result.validation_passed),
        "validation_errors": result.validation_errors if result else [],
        "iterations_used": result.iterations_used if result else 0,
        "status": result.status if result else "error",
        "duration_seconds": result.duration_seconds if result else duration,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "error": error_message,
    }
    records.append(rec)
    write_records(records)

    if error_message is None:
        marker = "✓" if rec["validation_passed"] else "✗"
        print(
            f"{marker} iters={rec['iterations_used']} "
            f"tokens=({input_tokens:>5},{output_tokens:>4}) "
            f"{duration:5.1f}s"
        )

print()
print(f"Done. {len(records)} records in {RECORDS_OUT}")

## Run-level summary

A glance at the run: how many translations validated, total tokens consumed, average duration.

In [ ]:
import pandas as pd

df = pd.DataFrame(records)
print(f"Total records:       {len(df)}")
print(f"Validation passed:   {df['validation_passed'].sum()} / {len(df)}")
print(f"Total input tokens:  {df['input_tokens'].sum():,}")
print(f"Total output tokens: {df['output_tokens'].sum():,}")
print(f"Mean duration:       {df['duration_seconds'].mean():.2f}s")
print()
df.groupby(["schema_name", "dialect"]).agg(
    n=("query_id", "count"),
    passed=("validation_passed", "sum"),
    mean_iter=("iterations_used", "mean"),
    total_input=("input_tokens", "sum"),
    total_output=("output_tokens", "sum"),
)

Records written. Proceed to `02_behavioural_metrics.ipynb`.